# NAS Search Space Experiment Notebook

Colab-compatible notebook for experimenting with the heterogeneous NCHW NAS search space.
Covers: family inference, architecture sampling, AZ-NAS proxy scoring, aging evolution.

**Run on Colab**: Runtime > Change runtime type > GPU (T4 recommended)

## 1. Setup

In [ ]:
import subprocess, sys

# --- On Colab: clone the repo and install deps ---
try:
    import google.colab
    IN_COLAB = True
    subprocess.check_call(['git', 'clone', '--depth=1',
        'https://github.com/jesusllg/UnseenNAS-AutoML-Competition', '/content/repo'])
    sys.path.insert(0, '/content/repo/submission')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'torch', 'torchvision', 'numpy', 'matplotlib', 'scipy'])
except ImportError:
    IN_COLAB = False
    import os
    # Assumes notebook is run from repo root
    repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
    if repo_root not in sys.path:
        sys.path.insert(0, os.path.join(repo_root, 'submission'))

print('IN_COLAB:', IN_COLAB)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time, random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}  |  device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Synthetic datasets for each family

In [ ]:
# Representative geometry for each family (C, H, W, num_classes)
SYNTHETIC_DATASETS = {
    'anisotropic':        (1,  6,  768, 20),   # Cryptic-like
    'small_grid':         (12, 8,  8,   64),   # Chesseract-like
    'possible_voxel':     (20, 20, 20,  10),   # Voxel-like
    'channel_heavy':      (8,  16, 16,  30),   # multi-channel mid-res
    'spatiotemporal_like':(3,  1,  64,  10),   # 1D sequence as 2D
    'visual_large':       (3,  128,128, 10),   # Myofibre-like
    'visual_medium':      (3,  32, 32,  10),   # CIFAR-like
    'compact_general':    (1,  16, 16,  5),
}

BATCH_SIZE = 16

def make_batch(C, H, W, num_classes, bs=BATCH_SIZE):
    x = torch.randn(bs, C, H, W)
    y = torch.randint(0, num_classes, (bs,))
    return x, y

print('Synthetic datasets ready')
for name, (C,H,W,NC) in SYNTHETIC_DATASETS.items():
    x, y = make_batch(C, H, W, NC)
    print(f'  {name:<22}  C={C}  H={H}  W={W}  classes={NC}  batch={x.shape}')

## 3. Family inference

In [ ]:
from search_space import infer_family

print(f'{'Dataset':<22}  {'Inferred family':<22}  max_pools  groupnorm  attn')
print('-' * 80)
for name, (C, H, W, NC) in SYNTHETIC_DATASETS.items():
    fp = infer_family(C, H, W, NC)
    match = '✓' if fp.name == name else f'→ {fp.name}'
    print(f'  {name:<22}  {fp.name:<22}  {fp.max_pool_steps}'
          f'          {str(fp.force_groupnorm):<5}      {fp.enable_attention}')

## 4. Sampling random architectures and repair

In [ ]:
from search_space import sample_random_genotype, repair, estimate_params

# Show 5 random architectures for visual_medium family
C, H, W, NC = 3, 32, 32, 10
family = infer_family(C, H, W, NC)
print(f'Family: {family.name}')
print()

random.seed(42)
for i in range(5):
    g = sample_random_genotype()
    g = repair(g, C, H, W, NC, family)
    params = estimate_params(g, C, H, W, NC)
    stage_info = [(g.stages[j].block_type[:6], g.stages[j].channels_idx)
                  for j in range(g.n_stages)]
    print(f'Arch {i+1}: n_stages={g.n_stages}  head={g.head_type:<15}'
          f'  norm={g.norm_type}  params≈{params/1e6:.2f}M')
    print(f'        stages: {stage_info}')

## 5. Build models and inspect architecture

In [ ]:
from search_space import build_model

C, H, W, NC = 3, 32, 32, 10
family = infer_family(C, H, W, NC)

random.seed(0)
g = sample_random_genotype()
g = repair(g, C, H, W, NC, family)

model = build_model(g, C, H, W, NC)
model.eval()

x = torch.randn(4, C, H, W)
with torch.no_grad():
    out = model(x)
print(f'Output shape: {out.shape}')

feats = model.extract_layer_features(x)
print(f'Layer features ({len(feats)} layers):')
for i, f in enumerate(feats):
    print(f'  Layer {i}: {tuple(f.shape)}')

total_params = sum(p.numel() for p in model.parameters())
print(f'Total params: {total_params:,}')
print(f'Model: {model}')

## 6. AZ-NAS Proxy Evaluation

In [ ]:
from search_space import az_nas_score_full

# Evaluate 15 random architectures, display scores
C, H, W, NC = 3, 32, 32, 10
family = infer_family(C, H, W, NC)
x = torch.randn(BATCH_SIZE, C, H, W)

random.seed(1)
results = []
for i in range(15):
    g = sample_random_genotype()
    g = repair(g, C, H, W, NC, family)
    try:
        model = build_model(g, C, H, W, NC)
        scores = az_nas_score_full(model, x, device, reinit=True)
        params = sum(p.numel() for p in model.parameters())
        results.append({'arch': i, 'params': params, **scores})
        print(f'  Arch {i+1:>2}: E={scores["expressivity"]:7.3f}  '
              f'P={scores["progressivity"]:7.3f}  '
              f'T={scores["trainability"]:7.3f}  '
              f'az={scores["az_nas"]:8.3f}  params={params/1e6:.1f}M')
    except Exception as e:
        print(f'  Arch {i+1:>2}: FAILED — {e}')

# Rank by AZ-NAS score
results.sort(key=lambda r: r['az_nas'], reverse=True)
print(f'\nTop architecture: Arch {results[0]["arch"]+1}  az_nas={results[0]["az_nas"]:.3f}')

In [ ]:
# Visualise score distributions
if results:
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, key, color in zip(axes,
            ['expressivity', 'progressivity', 'trainability', 'az_nas'],
            ['steelblue', 'darkorange', 'forestgreen', 'crimson']):
        vals = [r[key] for r in results if np.isfinite(r[key])]
        ax.hist(vals, bins=8, color=color, alpha=0.8, edgecolor='white')
        ax.set_title(key)
        ax.set_xlabel('Score')
    plt.suptitle('AZ-NAS proxy score distributions (15 random architectures)', y=1.02)
    plt.tight_layout()
    plt.show()

## 7. Aging Evolution Loop

In [ ]:
from search_space import aging_evolution, best_individual, az_nas_score

C, H, W, NC = 3, 32, 32, 10
family = infer_family(C, H, W, NC)
proxy_batch = torch.randn(BATCH_SIZE, C, H, W)

def proxy_fn(model, batch_x, dev):
    return az_nas_score(model, batch_x, dev, reinit=True)

print('Running Aging Evolution...')
t0 = time.time()
population = aging_evolution(
    family          = family,
    C               = C, H=H, W=W,
    num_classes     = NC,
    proxy_fn        = proxy_fn,
    batch_x         = proxy_batch,
    device          = device,
    n_population    = 20,
    n_rounds        = 50,
    tournament_size = 5,
    time_budget_s   = 120,   # 2 minute cap for notebook
    verbose         = False,
)
elapsed = time.time() - t0

best = best_individual(population)
print(f'\nEvolution complete in {elapsed:.1f}s')
print(f'Population size: {len(population)}')
print(f'Best fitness: {best.fitness:.4f}')
print(f'Best genotype: n_stages={best.genotype.n_stages}  '
      f'head={best.genotype.head_type}  norm={best.genotype.norm_type}')

In [ ]:
# Plot fitness distribution of final population
if population:
    fitnesses = sorted([ind.fitness for ind in population if np.isfinite(ind.fitness)], reverse=True)
    plt.figure(figsize=(8, 4))
    plt.bar(range(len(fitnesses)), fitnesses, color='steelblue', alpha=0.8)
    plt.xlabel('Architecture rank')
    plt.ylabel('AZ-NAS score')
    plt.title('Final population fitness (sorted)')
    plt.tight_layout()
    plt.show()

## 8. Proxy score vs. short training accuracy

In [ ]:
# Evaluate correlation between AZ-NAS proxy and 3-epoch training accuracy
from torch.utils.data import DataLoader, TensorDataset

C, H, W, NC = 3, 32, 32, 10
family = infer_family(C, H, W, NC)
proxy_batch = torch.randn(BATCH_SIZE, C, H, W)

# synthetic training dataset
n_train = 512
train_x = torch.randn(n_train, C, H, W)
train_y = torch.randint(0, NC, (n_train,))
train_dl = DataLoader(TensorDataset(train_x, train_y), batch_size=64, shuffle=True)
val_x = torch.randn(128, C, H, W)
val_y  = torch.randint(0, NC, (128,))

def short_train_acc(model, epochs=3):
    model.to(device).train()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = torch.nn.CrossEntropyLoss()
    for _ in range(epochs):
        for xb, yb in train_dl:
            opt.zero_grad()
            crit(model(xb.to(device)), yb.to(device)).backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        preds = model(val_x.to(device)).argmax(1).cpu()
    return (preds == val_y).float().mean().item()

print('Evaluating 10 architectures (proxy + 3-epoch training)...')
random.seed(42)
corr_data = []
for i in range(10):
    g = sample_random_genotype()
    g = repair(g, C, H, W, NC, family)
    try:
        model = build_model(g, C, H, W, NC)
        proxy = az_nas_score(model, proxy_batch, device, reinit=True)
        acc   = short_train_acc(build_model(g, C, H, W, NC))  # fresh init for training
        corr_data.append((proxy, acc))
        print(f'  Arch {i+1:>2}: proxy={proxy:8.3f}  val_acc={acc:.3f}')
    except Exception as e:
        print(f'  Arch {i+1:>2}: FAILED — {e}')

print(f'\nCollected {len(corr_data)} points')

In [ ]:
# Scatter plot: proxy score vs. training accuracy
if len(corr_data) >= 3:
    proxies, accs = zip(*corr_data)
    corr = np.corrcoef(proxies, accs)[0, 1]
    plt.figure(figsize=(6, 5))
    plt.scatter(proxies, accs, color='steelblue', s=80, alpha=0.8)
    plt.xlabel('AZ-NAS proxy score')
    plt.ylabel('Val accuracy (3 epochs)')
    plt.title(f'Proxy–accuracy correlation  r={corr:.3f}')
    # trend line
    z = np.polyfit(proxies, accs, 1)
    xs = np.linspace(min(proxies), max(proxies), 50)
    plt.plot(xs, np.poly1d(z)(xs), 'r--', alpha=0.6)
    plt.tight_layout()
    plt.show()
    print(f'Spearman-like correlation: {corr:.3f}')
else:
    print('Not enough data points for correlation plot.')

## 9. Integration test (end-to-end pipeline)

In [ ]:
# Simulate DataProcessor → NAS → Trainer → predict on synthetic data
import sys, os
# Ensure helpers module is accessible
submission_dir = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'submission')
if os.path.abspath(submission_dir) not in sys.path:
    sys.path.insert(0, os.path.abspath(submission_dir))

from torch.utils.data import DataLoader, TensorDataset

C, H, W, NC = 3, 32, 32, 10
n_total = 200

# Synthetic train/val/test splits
train_x = torch.randn(int(n_total*0.6), C, H, W)
train_y = torch.randint(0, NC, (int(n_total*0.6),))
val_x   = torch.randn(int(n_total*0.2), C, H, W)
val_y   = torch.randint(0, NC, (int(n_total*0.2),))
test_x  = torch.randn(int(n_total*0.2), C, H, W)
test_y  = torch.randint(0, NC, (int(n_total*0.2),))

train_dl = DataLoader(TensorDataset(train_x, train_y), batch_size=32, shuffle=True)
val_dl   = DataLoader(TensorDataset(val_x,   val_y),   batch_size=32)
test_dl  = DataLoader(TensorDataset(test_x,  test_y),  batch_size=32)

metadata = {
    'input_shape': [n_total, C, H, W],
    'num_classes': NC,
    'benchmark':   10.0,
    'codename':    'SyntheticTest',
    'time_limit':  0.05,  # 3 minutes
}

print('Metadata:', metadata)

In [ ]:
# Mock clock for integration test
class MockClock:
    def __init__(self, budget_seconds=180):
        self.start = time.perf_counter()
        self.budget = budget_seconds
    def check(self):
        return max(0.0, self.budget - (time.perf_counter() - self.start))

clock = MockClock(budget_seconds=180)

# NAS
from nas import NAS
print('Running NAS...')
t0 = time.time()
nas = NAS(train_dl, val_dl, metadata, clock)
model = nas.search()
print(f'NAS done in {time.time()-t0:.1f}s')
print(f'Model: {type(model).__name__}  params={sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Trainer
from trainer import Trainer

print('Training...')
t0 = time.time()
trainer = Trainer(model, device, train_dl, val_dl, metadata, clock)
trainer.train()
print(f'Training done in {time.time()-t0:.1f}s')

preds = trainer.predict(test_dl)
accuracy = (preds == test_y.numpy()).mean()
print(f'Test accuracy: {accuracy:.3f}  (random baseline: {1/NC:.3f})')
print('\nINTEGRATION TEST PASSED')

## 10. Multi-family sweep

In [ ]:
# Quick AZ-NAS sweep across all family geometries
print(f'{'Family':<22}  {'Best AZ-NAS':>12}  {'N archs':>8}  {'Time':>8}')
print('-' * 60)

sweep_results = {}
for fname, (C, H, W, NC) in list(SYNTHETIC_DATASETS.items())[:4]:  # first 4 to save time
    family = infer_family(C, H, W, NC)
    batch_x = torch.randn(min(BATCH_SIZE, 8), C, H, W)
    scores_local = []
    t0 = time.time()
    random.seed(99)
    for _ in range(8):
        g = sample_random_genotype()
        g = repair(g, C, H, W, NC, family)
        try:
            m = build_model(g, C, H, W, NC, aniso_axis=family.aniso_axis)
            s = az_nas_score(m, batch_x, device, reinit=True)
            if np.isfinite(s):
                scores_local.append(s)
        except Exception:
            pass
    elapsed = time.time() - t0
    best = max(scores_local) if scores_local else float('nan')
    sweep_results[fname] = scores_local
    print(f'  {fname:<22}  {best:>12.3f}  {len(scores_local):>8}  {elapsed:>7.1f}s')